In [31]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Treinamento do Modelo

No colab anterior, realizei as etapas de coleta, análise exploratória e processamento dos dados do csv. Agora, temos exatamente os dados limpos com tratamento para outliers.

Utilizaremos o modelo **RandomForestRegressor**.

Iremos separar 80% das viagens iniciais para treinar o modelo e os restantes 20% para testar a acurácia dele.



In [27]:
caminho_df_final = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados tratados/df_final.csv"
df = pd.read_csv(caminho_df_final)

#Convertendo a string de linha para um codigo numerico via label encoding
le_linha = LabelEncoder()

# 2. Ele lê a coluna 'servico' (texto), aprende as linhas e já cria a coluna 'linha' (número)
df['servico'] = le_linha.fit_transform(df['servico'])

display(df.head())
display(df.describe())
display(df.info())

,servico,sentido,duracao,mes_viagem,dia_viagem,hora_partida,dia_semana,eh_feriado
0,69,0,62.500000,2,25,361,2,0
1,227,0,70.983333,12,8,1103,6,0
2,61,1,105.500000,9,19,401,4,0
3,16,0,18.400000,6,27,293,4,0
4,65,0,132.000000,2,19,435,2,0


,servico,sentido,duracao,mes_viagem,dia_viagem,hora_partida,dia_semana,eh_feriado
count,222731.000000,222731.000000,222731.000000,222731.000000,222731.000000,222731.000000,222731.000000,222731.0
mean,131.657668,0.504874,61.121415,6.135235,15.655207,769.362523,2.689356,0.0
std,78.053173,0.499977,28.454268,3.619770,8.767187,330.155260,1.886149,0.0
min,0.000000,0.000000,5.000000,1.000000,1.000000,0.000000,0.000000,0.0
25%,65.000000,0.000000,40.100000,3.000000,8.000000,489.000000,1.000000,0.0
50%,127.000000,1.000000,59.083333,6.000000,15.000000,757.000000,3.000000,0.0
75%,201.000000,1.000000,78.550000,9.000000,23.000000,1042.000000,4.000000,0.0
max,280.000000,1.000000,474.933333,12.000000,31.000000,1439.000000,6.000000,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 222731 entries, 0 to 222730
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   servico       222731 non-null  int64  
 1   sentido       222731 non-null  int64  
 2   duracao       222731 non-null  float64
 3   mes_viagem    222731 non-null  int64  
 4   dia_viagem    222731 non-null  int64  
 5   hora_partida  222731 non-null  int64  
 6   dia_semana    222731 non-null  int64  
 7   eh_feriado    222731 non-null  int64  
dtypes: float64(1), int64(7)
memory usage: 13.6 MB


None

In [29]:

#os dados p treinar e respostas do modelo
X = df.drop(['duracao'], axis=1)
y = df['duracao']

#em seguida dividimos o dataset em 80% treinamento e 20% teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=19)
rfr = RandomForestRegressor(random_state=13)

#treinamento do modelo
rfr.fit(X_train, y_train)

#guardamos a previsao em uma variavel e depois avaliamos a acuracia do modelo
y_pred = rfr.predict(X_test)

print(mean_absolute_error(y_test, y_pred))
print(mean_squared_error(y_test, y_pred))
print(r2_score(y_test, y_pred))



7.074282408903954
117.47147279071281
0.8552219137175816


In [ ]:
#aqui iremos realizar uma calibração de hiperparametros do modelo acima

param_grid = {
    'n_estimators': [100, 200, 300],  # numero de arvores
    'max_depth': [ 10, 20, 30],  # profundidade maxima
    'min_samples_split': [2, 5, 10],  # numero minimo de amostras
    'min_samples_leaf': [1, 2, 4],  # numero minimo de amostras para um leaf node
}

#calibracao
rfr_cv = GridSearchCV(estimator=rfr, param_grid=param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)

#treinamento
rfr_cv.fit(X_train, y_train)

#avaliando
print(mean_absolute_error(y_test, y_pred))
print(mean_squared_error(y_test, y_pred))
print(r2_score(y_test, y_pred))